## Imports and Sentiment Tools

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

# sentiment tools
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from afinn import Afinn
from pysentimiento import create_analyzer
from transformers import pipeline

In [2]:
files = [f for f in Path('data_filtered/').iterdir() if f.is_file()]
files.reverse()
songs = pd.read_csv(files[0])

for file in files[1:]:
    data = pd.read_csv(file)
    songs = pd.concat([songs, data])

print(len(songs))
songs.head(3)

259458


,index,id,name,album_name,artists,danceability,energy,key,loudness,mode,...,duration_ms,lyrics,year,genre,popularity,total_artist_followers,avg_artist_popularity,artist_ids,niche_genres,is_valid_lyrics
0,256,3iBgrkexCzVuPy4O9vx7Mf,Glue Song,Glue Song,"[""beabadoobee""]",0.620,0.409,1,-10.146,1,...,135067,"i've never known someone like you, ooh tangled...",2023,Pop,81,6764580,80.000000,"[""35l9BRT7MXmM8bv2WDQiyB""]","[""bedroom pop""]",True
1,646,60SvhHtwefT0e2G7i7kOH3,New Gold (feat. Tame Impala and Bootie Brown),Cracker Island,"[""Gorillaz"", ""Tame Impala"", ""Bootie Brown""]",0.695,0.923,1,-3.930,0,...,215150,but in the magic cove there's a pretty one i a...,2023,Rock,77,24787609,78.333333,"[""3AA28KZvwAUcZuOKwyblJQ"", ""5INjqkS1o8h1imAzPq...","[""indie"", ""neo-psychedelic""]",True
2,891,0FA4wrjDJvJTTU8AepZTup,Watch This - ARIZONATEARS Pluggnb Remix,Watch This (ARIZONATEARS Pluggnb Remix),"[""Lil Uzi Vert"", ""sped up nightcore"", ""ARIZONA...",0.686,0.897,11,-7.180,0,...,163139,jump in a whip that you've never seen (yeah) i...,2023,Hip-Hop,75,19685914,71.333333,"[""4O15NlyKLIASxsJ0PrXPfz"", ""0M2CO5ijP35MDhNwvp...","[""melodic rap"", ""nightcore""]",True


In [3]:
def clean_lyrics(lyrics):
    cleaned = re.sub(r'\[.*?\]', '', lyrics)
    cleaned = re.sub(r'\d{1,2}:\d{2}(\.\d+)?', '', cleaned)
    cleaned = cleaned.strip()
    return cleaned

songs['lyrics'] = songs['lyrics'].apply(clean_lyrics)
songs = songs[songs['is_valid_lyrics'] == True].reset_index(drop=True)
print(f"Tracks entering sentiment pipeline: {len(songs):,}")

Tracks entering sentiment pipeline: 259,458


In [4]:
print(f'columns: {len(songs.columns)}')
songs.dtypes.rename('dtypes').to_frame()

columns: 26


,dtypes
index,int64
id,str
name,str
album_name,str
artists,str
danceability,float64
energy,float64
key,int64
loudness,float64
mode,int64


---
## Sentiment Tool Testing

- VADER
    - baseline/primary sentiment tool that auto gives continuous score [-1, 1]
- AFINN
    - lexicon-based sentiment tool similar to VADER but produces summed/averaged across text, NOT same range as VADER
- pysentimiento
    - toolkit for sentiment analysis like VADER/AFINN that outputs NEU, POS, NEG values (still need to calc)
- siebert/sentiment-roberta-large-english
    - classification model (can be used as directional/confidence check for VADER)

In [12]:
vader = SentimentIntensityAnalyzer()
afinn = Afinn()
sentimiento = create_analyzer(task="sentiment", lang="en")

i = 500
print(f'VALENCE: {songs['valence'].iloc[i]}')
test_lyric = songs['lyrics'].iloc[i]
test_lyric

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9420.66it/s]


VALENCE: 0.596


"반복된 day and night 갇힌 suit and tie 숨막힌 틀에 박힌 하루 막 뛰는 심장이 말해 chase the light 주저 말고 take it so good, good (everybody say) trippin', trippin', trippin' for fun drippin', drippin' all night long 짜릿한 기분 like it 커진 숨소릴 따라 ridin' trippin', trippin', trippin' for fun drippin', drippin' all night long 지금 이 리듬 like it 자유로운 느낌에 divin' look around me (꿈이 아닌 걸 believe it) see my heart waves (터질 듯한 느낌 feel it) 좀 더 빨리 어서 빨리 i need you closer 마주치는 눈빛 속 내 맘이 burning up 짜릿해져 이 순간 you make me feel so groovy 리듬에 body body movin' 이대로 going crazy for you 빠져들어 이 move keepin' it, keepin' it groovy keepin' it, keepin' it groovy 우리 둘만의 something you make me feel groovy ooh-ooh-ooh, ooh-ooh-ooh yea, you know we never gon' stop bring the beat back 너를 품에 안은 채로 돌리고 lean back 너와 나의 groove 마저 lay back back, back 너를 뒤에 두고 느껴봐 yeah, chillin' 여기 달빛 아래 우리 groovy drop it trippin', trippin', trippin' for fun drippin', drippin' all night long 지금 이 리듬 like it 자유로운 느낌에 divin' like a spotlight (나를 사로잡는 movin') think i'm fallin

In [13]:
vader_score = vader.polarity_scores(test_lyric)['compound']
vader_score

0.9828

In [14]:
def afinn_normalized_matched(text, afinn_obj):
    words = text.split()
    matched = [w for w in words if afinn_obj.score(w) != 0]
    if not matched:
        return 0.0
    raw_score = afinn_obj.score(text)
    return raw_score / len(matched)

afinn_score = afinn_normalized_matched(test_lyric, afinn)
afinn_score # unscaled afinn score

1.1111111111111112

In [15]:
def sentimiento_score(text, analyzer):
    result = analyzer.predict(text)
    probas = result.probas
    print(probas)
    # POS pulls toward +1, NEG pulls toward -1, NEU contributes 0
    score = probas.get('POS', 0) - probas.get('NEG', 0)
    return score

sentimiento_score(test_lyric, sentimiento)

{'NEG': 0.002495120046660304, 'NEU': 0.1948874443769455, 'POS': 0.8026174902915955}


0.8001223702449352

In [ ]:
sentiment_analysis = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")
sentiment_analysis(
    test_lyric,
    truncation=True,
    max_length=512
)

---
## Sentiment Features on Samples

In [ ]:
sample_songs = songs.sample(n=2000).copy()

In [ ]:
vader = SentimentIntensityAnalyzer()

sample_songs['vader_compound'] = sample_songs['lyrics'].apply(
    lambda x: vader.polarity_scores(x)['compound']
)
sample_songs[['valence', 'vader_compound']].describe()

In [ ]:
afinn = Afinn()

def afinn_normalized_matched(text, afinn_obj):
    words = text.split()
    matched = [w for w in words if afinn_obj.score(w) != 0]
    if not matched:
        return 0.0
    raw_score = afinn_obj.score(text)
    return raw_score / len(matched)

# raw score [-5, 5]
sample_songs['afinn_raw'] = sample_songs['lyrics'].apply(lambda x: afinn_normalized_matched(x, afinn))

# rescale to [-1, 1] for rough comparability with VADER
min_val, max_val = sample_songs['afinn_raw'].min(), sample_songs['afinn_raw'].max()
sample_songs['afinn_rescaled'] = 2 * (sample_songs['afinn_raw'] - min_val) / (max_val - min_val) - 1
sample_songs[['valence', 'afinn_rescaled']].describe()

In [ ]:
sentimiento = create_analyzer(task="sentiment", lang="en")

samples_for_sentimiento = sample_songs.sample(n=(len(sample_songs) // 10), random_state=42).copy()
samples_for_sentimiento['pysentimiento_score'] = samples_for_sentimiento ['lyrics'].apply(
    lambda x: sentimiento_score(x, sentimiento)
)
samples_for_sentimiento[['valence', 'pysentimiento_score']].describe()

In [ ]:
from tqdm import tqdm
tqdm.pandas()

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    device=-1,
    truncation=True,
    max_length=512
)

def get_signed_roberta_score(text):
    result = sentiment_pipe(text)[0]
    return result['score'] if result['label'] == 'POSITIVE' else -result['score']

# NOTE: run on a sample first if the full dataset is large — this is CPU and slow
sample_for_roberta = sample_songs.sample(n=(len(sample_songs) // 10), random_state=42).copy()
sample_for_roberta['roberta_signed'] = sample_for_roberta['lyrics'].progress_apply(get_signed_roberta_score)
sample_for_roberta['roberta_signed'].describe()

In [ ]:
from scipy.stats import pearsonr, spearmanr

merged_check = samples_for_sentimiento[['vader_compound', 'afinn_rescaled', 'pysentimiento_score']].dropna()

pairs = [('vader_compound', 'afinn_rescaled'), ('vader_compound', 'pysentimiento_score'),
        #  ('vader_compound', 'roberta_signed'), 
         ('afinn_rescaled', 'pysentimiento_score')]

for a, b in pairs:
    r, _ = pearsonr(merged_check[a], merged_check[b])
    rho, _ = spearmanr(merged_check[a], merged_check[b])
    print(f"{a} vs {b}: Pearson r={r:.3f}, Spearman rho={rho:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

sns.histplot(sample_songs['valence'], ax=axes[0], bins=40, color='steelblue')
axes[0].set_title('Valence')

sns.histplot(sample_songs['vader_compound'], ax=axes[1], bins=40, color='steelblue')
axes[1].set_title('VADER Compound')

sns.histplot(sample_songs['afinn_rescaled'], ax=axes[2], bins=40, color='salmon')
axes[2].set_title('AFINN Rescaled')

sns.histplot(samples_for_sentimiento['pysentimiento_score'], ax=axes[3], bins=40, color='seagreen')
axes[3].set_title('RoBERTa Signed (sample)')

# sns.histplot(sample_for_roberta['roberta_signed'], ax=axes[2], bins=40, color='seagreen')
# axes[2].set_title('RoBERTa Signed (sample)')

plt.tight_layout()
plt.show()